# So sánh 3 kiến trúc PhoBERT NER: softmax vs CRF vs LSTM-CRF

Notebook này train **crf** và **lstm_crf** trên đúng bộ dữ liệu đã dùng cho softmax (`data_dir/` có sẵn trong repo,
**không sinh lại** để so sánh công bằng), rồi gộp kết quả softmax (đã có trên Drive) + baseline từ điển thành một bảng/biểu đồ.

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Mỗi kiến trúc ~15–30 phút trên T4.

## Research questions (RQ)
- **RQ1.** PhoBERT có vượt baseline từ điển + regex không, nhất là trên câu ngoài từ điển (`test_hard`)?
- **RQ2.** Thêm CRF hoặc LSTM-CRF lên trên softmax có cải thiện kết quả không?
- **RQ3.** Câu không dấu và câu có phủ định ảnh hưởng đến độ chính xác như thế nào?

Mỗi phần kết quả ở mục 6 ghi rõ nó trả lời RQ nào; ô **"Kết luận"** sau mỗi phần để bạn tự điền sau khi có số.

In [ ]:
#@title 1. Clone repo + cd vào project
BRANCH = "main"  #@param {type:"string"}

!git clone -b {BRANCH} https://github.com/letuanlocc/training_model.git /content/training_model
%cd /content/training_model/phobert_ner
!ls data_dir

In [ ]:
#@title 2. Cài môi trường (Java + Python packages + VnCoreNLP)
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null

# giữ torch CUDA có sẵn của Colab, chỉ cài các gói còn lại
!grep -v '^#' requirements_working.txt | grep -v -E '^(numpy|pandas|matplotlib|scikit-learn)' | sed '/^$/d' > /tmp/req_colab.txt
!pip -q install -r /tmp/req_colab.txt
!pip -q install py_vncorenlp

import os
VN = os.path.abspath('vncorenlp')   # = phobert_ner/vncorenlp (đúng đường dẫn mà code của repo tìm)
B = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master'
os.makedirs(f'{VN}/models/wordsegmenter', exist_ok=True)
for f in ['VnCoreNLP-1.2.jar', 'VnCoreNLP-1.1.1.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
    !wget -q -nc -O {VN}/{f} {B}/{f}
print('Xong.')

In [ ]:
#@title 3. Kiểm tra dữ liệu (không sinh lại) + thống kê
!python data_build/validate_data.py | tail -15

import collections, pandas as pd
def stats(path):
    n_sent, ents = 0, collections.Counter()
    for block in open(path, encoding='utf-8').read().split('\n\n'):
        rows = [l.split('\t') for l in block.splitlines() if l.strip()]
        if not rows: continue
        n_sent += 1
        for r in rows:
            if r[-1].startswith('B-'): ents[r[-1][2:]] += 1
    return dict(cau=n_sent, **{k: ents[k] for k in ['SKILL', 'JOB', 'TIME', 'LOC']})
pd.DataFrame({s: stats(f'data_dir/{s}.txt') for s in ['train', 'dev', 'test', 'test_hard']}).T

## 4. Train các kiến trúc

Mặc định chỉ train crf + lstm_crf vì softmax đã có kết quả trên Drive.
Script `train_<arch>.sh` tự train → predict → ghi `outputs/<arch>/report_test.txt` và `report_test_hard.txt`.
**Không chạy 2 train song song** (dùng chung file cache trong `data_dir/`).

In [ ]:
#@title 4a. Train crf
!bash train_crf.sh 2>&1 | grep -v 'it/s]'

In [ ]:
#@title 4b. Train lstm_crf
!bash train_lstm_crf.sh 2>&1 | grep -v 'it/s]'

In [ ]:
#@title 4c. (Tuỳ chọn) Train lại softmax — chỉ khi muốn tái lập, bình thường bỏ qua
RETRAIN_SOFTMAX = False  #@param {type:"boolean"}
if RETRAIN_SOFTMAX:
    !bash train_softmax.sh 2>&1 | grep -v 'it/s]'

## 5. Lưu model lên Drive (tránh mất khi Colab ngắt kết nối)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DEST = "/content/drive/MyDrive/phobert_ner_outputs"
!mkdir -p "{DEST}"
for arch in ['crf', 'lstm_crf']:
    if os.path.isdir(f'outputs/{arch}'):
        !cp -r outputs/{arch} "{DEST}/{arch}_main"
        print('Đã lưu', arch)

## 6. Gộp kết quả: softmax (Drive) + baseline + crf + lstm_crf
Báo cáo của softmax lấy từ thư mục Drive chung. Nếu không tải được thì copy 2 file `report_test.txt`,
`report_test_hard.txt` vào `outputs/softmax/` thủ công.

In [ ]:
#@title 6a. Lấy báo cáo softmax + chạy baseline
!pip -q install gdown
os.makedirs('outputs/softmax', exist_ok=True)
SOFTMAX_FILES = {  # id file trong thư mục Drive softmax_main
    'report_test.txt': '18Uc0Od1PJOxEo_yqrBxqudpDkTVUzZXj',
    'report_test_hard.txt': '1UfUzRuQ1cCKNyMv_uoUn9jhJuWxKnILV',
    'pred_test.txt': '1Od6kt0tY4_hbaGmSZp-uQBFCn2ASaBEB',
    'pred_test_hard.txt': '1tUgMhlNo7_s7LvSG0B65j9JeJh-b-ha_',
}
for name, fid in SOFTMAX_FILES.items():
    if not os.path.exists(f'outputs/softmax/{name}'):
        !gdown -q "https://drive.google.com/uc?id={fid}" -O outputs/softmax/{name}

# baseline từ điển + regex: sinh dự đoán rồi chấm bằng cùng eval_report.py
os.makedirs('outputs/baseline', exist_ok=True)
!python data_build/baseline_dict.py > /dev/null
for split in ['test', 'test_hard']:
    !python data_build/eval_report.py data_dir/preds/baseline_{split}.txt data_dir/{split}_meta.jsonl > outputs/baseline/report_{split}.txt
!ls outputs/*/report_*.txt

### RQ1 + RQ2 — bảng so sánh kiến trúc
- **RQ1:** so dòng `baseline` với các dòng model, đặc biệt `split = test_hard`.
- **RQ2:** so `softmax` với `crf` và `lstm_crf` (cùng dữ liệu, cùng siêu tham số).

In [ ]:
#@title 6b. Bảng so sánh
import re, pandas as pd

ROW = re.compile(r'^\s*(JOB|LOC|SKILL|TIME|micro avg)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)', re.M)

def parse(path):
    txt = open(path, encoding='utf-8').read()
    res = {}
    for m in ROW.finditer(txt):
        res.setdefault(m.group(1), dict(P=float(m.group(2)), R=float(m.group(3)), F1=float(m.group(4))))
    return res

ARCHS = ['baseline', 'softmax', 'crf', 'lstm_crf']
rows = []
for arch in ARCHS:
    for split in ['test', 'test_hard']:
        p = f'outputs/{arch}/report_{split}.txt'
        if not os.path.exists(p):
            print('Thiếu', p); continue
        r = parse(p)
        rows.append(dict(model=arch, split=split, micro_P=r['micro avg']['P'], micro_R=r['micro avg']['R'],
                         micro_F1=r['micro avg']['F1'], **{f'F1_{k}': r[k]['F1'] for k in ['JOB', 'LOC', 'SKILL', 'TIME']}))
df = pd.DataFrame(rows).round(4)
df

In [ ]:
#@title 6c. Biểu đồ micro-F1 và F1 theo nhãn
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, split in zip(axes, ['test', 'test_hard']):
    d = df[df.split == split].set_index('model')[['F1_JOB', 'F1_LOC', 'F1_SKILL', 'F1_TIME', 'micro_F1']]
    d.columns = ['JOB', 'LOC', 'SKILL', 'TIME', 'micro']
    d.plot.bar(ax=ax, rot=0, width=0.8)
    ax.set_title(split); ax.set_ylim(0.5, 1.0); ax.set_xlabel(''); ax.grid(axis='y', alpha=.3)
axes[0].set_ylabel('F1 (mức thực thể)')
plt.tight_layout()
plt.savefig('outputs/compare.png', dpi=150)
plt.show()
df.to_csv('outputs/compare.csv', index=False)

**Kết luận RQ1 (điền sau khi có số):** ...

**Kết luận RQ2 (điền sau khi có số):** ... (lưu ý: 1 seed/kiến trúc, chênh lệch < ~0.01 có thể chỉ là nhiễu)

### RQ3 — ảnh hưởng của câu không dấu và câu phủ định
Bảng dưới là F1 theo nhóm câu. So cột nhóm `không dấu` / `phủ định` với `có dấu` ở từng model.

In [ ]:
#@title 6d. Theo nhóm câu: có dấu / không dấu / phủ định
GRP = re.compile(r'^(có dấu|không dấu|phủ định)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)', re.M)
rows = []
for arch in ARCHS:
    for split in ['test', 'test_hard']:
        p = f'outputs/{arch}/report_{split}.txt'
        if os.path.exists(p):
            for m in GRP.finditer(open(p, encoding='utf-8').read()):
                rows.append(dict(model=arch, split=split, nhom=m.group(1), F1=float(m.group(5))))
g = pd.DataFrame(rows).pivot_table(index=['split', 'nhom'], columns='model', values='F1')
g[[a for a in ARCHS if a in g.columns]]

**Kết luận RQ3 (điền sau khi có số):** ...

## 7. Phân tích lỗi
Với mỗi model, so thực thể gold và thực thể dự đoán trong `pred_<split>.txt` rồi phân loại lỗi:
- **thiếu** (missed): thực thể gold không có dự đoán nào chồng lên
- **thừa** (spurious): thực thể dự đoán không chồng lên thực thể gold nào
- **sai ranh giới**: cùng nhãn, có chồng lên nhau nhưng khác vị trí bắt đầu/kết thúc
- **sai nhãn**: chồng lên nhau nhưng khác nhãn

In [ ]:
#@title 7a. Đếm lỗi theo loại
def read_pred_file(path):
    sents = []
    for block in open(path, encoding='utf-8').read().split('\n\n'):
        rows = [l.rstrip('\r').split('\t') for l in block.splitlines() if l.strip()]
        if rows:
            sents.append(([r[0] for r in rows], [r[-2] for r in rows], [r[-1] for r in rows]))
    return sents

def spans(tags):
    out, start, lab = [], None, None
    for i, t in enumerate(tags + ['O']):
        if t == 'O' or t.startswith('B-') or (t.startswith('I-') and t[2:] != lab):
            if start is not None:
                out.append((lab, start, i)); start, lab = None, None
            if t != 'O':
                start, lab = i, t[2:]
    return out

def classify(gold, pred):
    errs = []
    g, p = spans(gold), spans(pred)
    for gs in g:
        ov = [ps for ps in p if ps[1] < gs[2] and gs[1] < ps[2]]
        if not ov: errs.append(('thiếu', gs, None))
        elif gs not in ov:
            same = [ps for ps in ov if ps[0] == gs[0]]
            errs.append(('sai ranh giới' if same else 'sai nhãn', gs, ov[0]))
    for ps in p:
        if not any(ps[1] < gs[2] and gs[1] < ps[2] for gs in g):
            errs.append(('thừa', None, ps))
    return errs

summary, details = [], {}
for arch in ARCHS:
    for split in ['test', 'test_hard']:
        path = f'outputs/{arch}/pred_{split}.txt' if arch != 'baseline' else f'data_dir/preds/baseline_{split}.txt'
        if not os.path.exists(path): continue
        cnt = collections.Counter(); miss_by_label = collections.Counter()
        for si, (toks, gold, pred) in enumerate(read_pred_file(path)):
            for kind, gs, ps in classify(gold, pred):
                cnt[kind] += 1
                if kind == 'thiếu': miss_by_label[gs[0]] += 1
                details.setdefault((arch, split), []).append((si, kind, toks, gold, pred, gs, ps))
        summary.append(dict(model=arch, split=split, **{k: cnt[k] for k in ['thiếu', 'thừa', 'sai ranh giới', 'sai nhãn']}, tong=sum(cnt.values())))
pd.DataFrame(summary)

In [ ]:
#@title 7b. Xem ví dụ câu sai
MODEL = "softmax"  #@param ["baseline", "softmax", "crf", "lstm_crf"]
SPLIT = "test_hard"  #@param ["test", "test_hard"]
N = 15  #@param {type:"integer"}
for si, kind, toks, gold, pred, gs, ps in details.get((MODEL, SPLIT), [])[:N]:
    g = ' '.join(toks[gs[1]:gs[2]]) + f' [{gs[0]}]' if gs else '-'
    p = ' '.join(toks[ps[1]:ps[2]]) + f' [{ps[0]}]' if ps else '-'
    print(f"[{kind}] câu: {' '.join(toks)}\n    gold: {g}   |   dự đoán: {p}\n")

**Nhận xét lỗi (điền sau khi xem 7a, 7b):** nhãn nào hay thiếu/thừa nhất, lỗi tập trung ở loại câu nào
(không dấu, phủ định, thực thể ngoài từ điển...), nguyên nhân có thể (ranh giới cụm TIME, từ "giỏi/thành thạo"...).

In [ ]:
#@title 8. Lưu bảng + biểu đồ lên Drive
!cp outputs/compare.csv outputs/compare.png "{DEST}/"
print('Đã lưu compare.csv / compare.png vào', DEST)

---
### Lưu ý
- Cả 3 kiến trúc dùng cùng `data_dir/`, cùng siêu tham số (xem `train_*.sh`) → so sánh công bằng. Mỗi kiến trúc chỉ train **1 lần, 1 seed**, nên chênh lệch nhỏ (< ~0.01) có thể chỉ là nhiễu.
- `ModuleNotFoundError` / `vncorenlp jar not found` / runtime bị reset → chạy lại cell 1 và 2.
- Lỗi `mask of the first timestep must all be on` → dùng nhầm cache của kiến trúc khác; các script đã có `--overwrite_data`, đừng chạy song song.